# Closing the remaining data gaps

The core pipeline (`ml/bits_ml`: ingest → grid → rain_panel → panel → seasons) leaves four gaps.
This notebook closes the ones the data can close and reports the ones it cannot.

| # | gap | before | fix |
|---|---|---|---|
| 1 | specific yield missing from the raw file | measured for 2,880 wells, the rest guessed from state medians | read every well from the hydrogeological map the published values came from |
| 2 | aquifer type blank ("-", "Not available") | 1,697 of 8,629 modelling wells | dug wells → Unconfined (the data shows 99.6% are), the rest → `Unknown` |
| 2b | well type blank ("-") | 2,239 wells | `Unknown` |
| 3 | monthly and season tables carry the old specific yield | — | rebuild them |
| 4 | no cell × season table with the wells in it | monthly only | build `fact_cell_season_full` |
| 5 | edge cases | — | reported, not changed |

**Run order.** After the core pipeline, and again after any re-run of `python -m bits_ml.ingest`
(which rewrites `well_master`). Running it twice gives the same result: the file's original
values are kept in `*_cgwb` / `*_ingest` columns, and every fix is derived from those.

Everything is written to `data_cleaning/data/`. Why each change was made is in `README.md`;
what each table is for is in `DATASETS.md`.

In [1]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile
from scipy import ndimage

HERE = Path.cwd() if Path.cwd().name == "data_cleaning" else Path.cwd() / "data_cleaning"
REPO = HERE.parent
sys.path.insert(0, str(REPO / "ml"))

from bits_ml import panel, seasons
from bits_ml.config import CGWB_ZIP, CORE_DIR, HYDRO_MAP_MEMBER

pd.set_option("display.width", 140)
wells = pd.read_parquet(CORE_DIR / "well_master.parquet")

# Keep the values the ingest produced, so every fix below starts from them and a re-run is harmless.
for column, keep in (("aquifer", "aquifer_cgwb"), ("well_type", "well_type_cgwb"),
                     ("sy", "sy_ingest"), ("sy_source", "sy_source_ingest")):
    if keep not in wells:
        wells[keep] = wells[column]

model = wells["view_modelling"]
print(f"wells {len(wells):,} | modelling view {int(model.sum()):,}")

wells 32,299 | modelling view 8,629


## Gap 1 — specific yield from the hydrogeological map

Specific yield (Sy) is the share of the rock that holds water. It turns a change in water level
into millimetres of water. The raw CGWB file has no Sy. The source paper sampled a
hydrogeological map at its 2,759 wells and published the result, and the ingest filled every
other well with the median of its state and aquifer type: a guess.

The map is in the archive. It is a 0.12° grid (about 13 km) of five Sy classes, one per rock
type. First check: does sampling it at the published wells give back the published values?

In [2]:
with zipfile.ZipFile(CGWB_ZIP) as z, MemoryFile(z.read(HYDRO_MAP_MEMBER)) as mem, mem.open() as src:
    sy_grid = src.read(1).astype(float)
    sy_grid[sy_grid < -1e30] = np.nan            # nodata: sea and outside the map
    sy_grid = np.round(sy_grid, 3)               # stored as float32: 0.023 reads back as 0.02300000004
    transform = src.transform
print(f"map {sy_grid.shape[1]} x {sy_grid.shape[0]} cells at {transform.a}°, "
      f"classes {sorted(np.unique(sy_grid[np.isfinite(sy_grid)]).round(3).tolist())}")

ROCK = {0.018: "folded metasediments / sedimentary aquitard",
        0.020: "jointed crystalline",
        0.023: "fractured crystalline",
        0.043: "consolidated sedimentary, permeable",
        0.130: "unconsolidated sedimentary (alluvium)"}

rows, cols = rasterio.transform.rowcol(transform, wells["lon"].to_numpy(), wells["lat"].to_numpy())
rows, cols = np.asarray(rows), np.asarray(cols)
inside = (rows >= 0) & (rows < sy_grid.shape[0]) & (cols >= 0) & (cols < sy_grid.shape[1])
sy_map = np.full(len(wells), np.nan)
sy_map[inside] = sy_grid[rows[inside], cols[inside]]

published = (wells["sy_source_ingest"] == "published").to_numpy()
both = published & np.isfinite(sy_map)
agree = np.isclose(wells.loc[both, "sy_ingest"], sy_map[both]).mean()
print(f"published wells {published.sum():,} | on a map cell {both.sum():,} | map gives the published value for {agree:.1%}")
pd.crosstab(wells.loc[both, "sy_ingest"].round(3).rename("published"),
            pd.Series(sy_map[both], index=wells.index[both], name="map"))

map 243 x 241 cells at 0.12°, classes [0.018, 0.02, 0.023, 0.043, 0.13]
published wells 2,880 | on a map cell 2,872 | map gives the published value for 100.0%


map,0.018,0.020,0.023,0.043,0.130
published,,,,,
0.018,134,0,0,0,0
0.020,0,637,0,0,0
0.023,0,0,1211,0,0
0.043,0,0,0,292,0
0.130,0,0,0,0,598


The map reproduces the published values exactly, so it is the right source for every other well.

A well on the coast or the border can fall on an empty map cell. Those take the nearest map
cell up to two cells away (about 27 km), recorded as `hydro_map_nearest`. Beyond that, the
ingest's guess stays and says so.

In [3]:
MAX_NEAREST_CELLS = 2

empty = ~np.isfinite(sy_grid)
distance, (near_r, near_c) = ndimage.distance_transform_edt(empty, return_indices=True)
r, c = np.clip(rows, 0, sy_grid.shape[0] - 1), np.clip(cols, 0, sy_grid.shape[1] - 1)
nearest_ok = inside & ~np.isfinite(sy_map) & (distance[r, c] <= MAX_NEAREST_CELLS)
sy_near = np.where(nearest_ok, sy_grid[near_r[r, c], near_c[r, c]], np.nan)

source = np.select(
    [published, np.isfinite(sy_map), nearest_ok],
    ["published", "hydro_map", "hydro_map_nearest"],
    default=wells["sy_source_ingest"].to_numpy(),
)
sy_new = np.select([published, np.isfinite(sy_map), nearest_ok],
                    [wells["sy_ingest"].to_numpy(), sy_map, sy_near],
                    default=wells["sy_ingest"].to_numpy())

was_guess = ~published & np.isin(source, ["hydro_map", "hydro_map_nearest"])
moved = np.abs(sy_new - wells["sy_ingest"].to_numpy().round(3))
changed = was_guess & (moved > 1e-6)
print(f"guesses replaced by the map: {was_guess.sum():,} | Sy class changed for {changed.sum():,} ({changed.sum() / was_guess.sum():.0%}), "
      f"by a median {np.median(moved[changed]):.3f} and up to {moved[changed].max():.3f}")
ratio = sy_new[changed] / wells["sy_ingest"].to_numpy()[changed]
print(f"where it changed, the new Sy is off from the guess by more than half (x<0.67 or x>1.5) for {((ratio < 1/1.5) | (ratio > 1.5)).mean():.0%}")

wells["sy"] = sy_new
wells["sy_source"] = source
wells["rock_class"] = pd.Series(np.round(sy_new, 3)).map(ROCK).where(
    np.isin(source, ["published", "hydro_map", "hydro_map_nearest"])).to_numpy()

summary = pd.DataFrame({
    "all wells, before": wells["sy_source_ingest"].value_counts(),
    "all wells, after": wells["sy_source"].value_counts(),
    "modelling, before": wells.loc[model, "sy_source_ingest"].value_counts(),
    "modelling, after": wells.loc[model, "sy_source"].value_counts(),
}).fillna(0).astype(int)
summary

guesses replaced by the map: 29,419 | Sy class changed for 8,996 (31%), by a median 0.087 and up to 0.112
where it changed, the new Sy is off from the guess by more than half (x<0.67 or x>1.5) for 75%


,"all wells, before","all wells, after","modelling, before","modelling, after"
hydro_map,0,29307,0,6306
hydro_map_nearest,0,112,0,7
imputed_national,733,0,199,0
imputed_state,4586,0,191,0
imputed_state_aquifer,24100,0,5923,0
published,2880,2880,2316,2316


Where a guess and the map disagree, the difference is not small: Sy runs from 0.018 in hard
rock to 0.13 in alluvium, a factor of seven, and every millimetre of storage scales with it.

## Gap 2 — aquifer type and well type

`aquifer` says whether the well taps a free water table (unconfined) or water under a
confining layer. It is "-" or "Not available" for a fifth of the modelling wells. A dug well is
an open pit into the water table, so it should almost always be unconfined. The wells whose
type *is* recorded can confirm that:

In [4]:
BLANK = {"-", "Not available", "", "nan", "None"}
aq = wells["aquifer_cgwb"].where(~wells["aquifer_cgwb"].astype(str).str.strip().isin(BLANK))
wt = wells["well_type_cgwb"].where(~wells["well_type_cgwb"].astype(str).str.strip().isin(BLANK))

known = pd.crosstab(wt.fillna("Unknown"), aq, margins=True, margins_name="total")
known["share unconfined"] = (known["Unconfined"] / known["total"]).round(3)
known

aquifer_cgwb,Confined,Semi-confined,Unconfined,total,share unconfined
well_type_cgwb,,,,,
Bore well,166,671,1479,2316,0.639
Dug cum bore well,0,0,17,17,1.000
Dug well,17,35,13912,13964,0.996
Piezometer,15,30,154,199,0.774
Slim hole,0,0,1,1,1.000
Tube well,337,162,1986,2485,0.799
Unknown,0,3,2,5,0.400
total,535,901,17551,18987,0.924


Dug wells with a recorded type are 99.6% unconfined, and dug-cum-bore wells all are, so a blank
dug well is filled as Unconfined. Bore wells (64% unconfined), tube wells (80%) and piezometers (77%) are too mixed to
guess, so a blank there becomes `Unknown`. `aquifer_source` records
which rows were filled.

In [5]:
DUG = {"Dug well", "Dug cum bore well"}
fill = aq.isna() & wt.isin(DUG)

wells["aquifer"] = aq.where(~fill, "Unconfined").fillna("Unknown")
wells["aquifer_source"] = np.select([aq.notna(), fill], ["cgwb", "inferred_dug_well"], default="unknown")
wells["well_type"] = wt.fillna("Unknown")

pd.DataFrame({
    "all wells": wells["aquifer_source"].value_counts(),
    "modelling": wells.loc[model, "aquifer_source"].value_counts(),
}).fillna(0).astype(int)

,all wells,modelling
aquifer_source,,
cgwb,18987,6932
inferred_dug_well,6308,1229
unknown,7004,468


In [6]:
wells.to_parquet(CORE_DIR / "well_master.parquet", index=False)
print("modelling wells, aquifer:", wells.loc[model, "aquifer"].value_counts().to_dict())
print("modelling wells, Sy measured from the map or published:",
      f"{wells.loc[model, 'sy_source'].isin(['published', 'hydro_map', 'hydro_map_nearest']).mean():.1%}")

modelling wells, aquifer: {'Unconfined': 7669, 'Unknown': 468, 'Semi-confined': 393, 'Confined': 99}
modelling wells, Sy measured from the map or published: 100.0%


## Gap 3 — rebuild the tables that carry Sy and aquifer

`fact_well_month` and every season table copy `sy`, `sy_source`, `aquifer` and `well_type` from
`well_master`, and storage in millimetres is computed from Sy. They are rebuilt with the
pipeline's own code, not patched.

In [7]:
panel.main()
print()
seasons.main()

fact_well_month      rows 2,588,700 | wells 8,629 | with a level 579,364 (22% of rows, 67% of campaign rows)
                     rain missing 189 | anomaly missing where a level exists 0
fact_cell_month_full rows 1,489,200 | cells 4,964 | rows carrying wells 220,102
                     wells behind a cell reading: median 2, max 48, single-well 30%
                     mean vs idw differ by median 0.038 m, mean vs median 0.000 m
fact_district_month  rows   216,000 | districts 720 | rows carrying wells 40,893

wrote fact_well_month, fact_cell_month_full, fact_district_month to C:\Users\Public\BITS\BITS_grp18\data_cleaning\data

dim_season 99 seasons, 1998-01-16 to 2022-11-15
reading the daily grid:


fact_cell_season     rows   491,436 | cells 4,964 | incomplete cell-seasons (suspect zeros blanked) 3,458
                     mean rain per cell: Nov-Jan 30 mm, Jan-May 122 mm, May-Aug 609 mm, Aug-Nov 376 mm
fact_well_season     rows   854,271 | wells 8,629 | with a level 579,364 | with a change 564,852 (88% from the season just before)
                     Nov-Jan: median change +0.65 m, rain 7 mm, recharge ratio -0.089
                     Jan-May: median change +1.42 m, rain 39 mm, recharge ratio -0.562
                     May-Aug: median change -1.94 m, rain 482 mm, recharge ratio +0.116
                     Aug-Nov: median change +0.15 m, rain 359 mm, recharge ratio -0.013
fact_district_season rows    71,280 | districts 720 | district-seasons with a well 40,893

wrote dim_season, fact_cell_season, fact_well_season, fact_district_season to C:\Users\Public\BITS\BITS_grp18\data_cleaning\data


## Gap 4 — the cell × season table with its wells

The monthly layer has `fact_cell_month_full`: rain per grid cell with the wells standing in it
summarised beside it. The season layer only had rain per cell. `fact_cell_season_full` adds
the wells: how many were read, their median anomaly and how much they disagree, and their
median change and storage. A cell with one well is a much weaker claim than a cell with twenty,
so the count sits in every row.

In [8]:
well_season = pd.read_parquet(CORE_DIR / "fact_well_season.parquet")
cell_season = pd.read_parquet(CORE_DIR / "fact_cell_season.parquet")
dim_cell = pd.read_parquet(CORE_DIR / "dim_cell.parquet")
dim_season = pd.read_parquet(CORE_DIR / "dim_season.parquet")

read = well_season[well_season["depth_mbgl"].notna()]
in_cell = read.groupby(["cell_id", "season_idx"], as_index=False).agg(
    n_wells_read=("well_uid", "nunique"),
    anomaly_mean_m=("anomaly_ref_m", "mean"),
    anomaly_median_m=("anomaly_ref_m", "median"),
    anomaly_spread_m=("anomaly_ref_m", "std"),
    depth_median_m=("depth_mbgl", "median"),
    n_wells_change=("delta_h_m", "count"),
    delta_h_median_m=("delta_h_m", "median"),
    storage_change_median_mm=("storage_change_mm", "median"),
    recharge_ratio_median=("recharge_ratio", "median"),
)

full = cell_season.merge(in_cell, on=["cell_id", "season_idx"], how="left")
full[["n_wells_read", "n_wells_change"]] = full[["n_wells_read", "n_wells_change"]].fillna(0).astype(int)
full = full.merge(dim_cell[["cell_id", "cell_lat", "cell_lon", "district", "state"]], on="cell_id", how="left")
full = full.merge(dim_season[["season_idx", "season_year", "season", "season_label", "water_year"]],
                  on="season_idx", how="left")
full = full.sort_values(["cell_id", "season_idx"], ignore_index=True)
full.to_parquet(CORE_DIR / "fact_cell_season_full.parquet", index=False)

with_wells = full[full["n_wells_read"] > 0]
print(f"fact_cell_season_full rows {len(full):,} | cells {full['cell_id'].nunique():,} "
      f"| cell-seasons with wells {len(with_wells):,} | single-well {(with_wells['n_wells_read'] == 1).mean():.0%}")
with_wells.groupby("season")[["rain_mm", "delta_h_median_m", "storage_change_median_mm"]].median().round(2)

fact_cell_season_full rows 491,436 | cells 4,964 | cell-seasons with wells 220,102 | single-well 31%


,rain_mm,delta_h_median_m,storage_change_median_mm
season,,,
Aug-Nov,325.87,0.17,-6.00
Jan-May,32.50,1.50,-49.68
May-Aug,465.74,-2.07,69.30
Nov-Jan,2.91,0.70,-23.00


## Gap 5 — edge cases, reported and left alone

These are either correct as they are or have nothing in the data to fix them with.

In [9]:
dist = pd.read_parquet(CORE_DIR / "bridge_cell_district.parquet")
land = dim_cell[dim_cell["is_land"]]
obs = pd.read_parquet(CORE_DIR / "well_obs.parquet", columns=["well_uid", "year", "campaign", "valid"])
obs = obs[(obs["year"] <= 2022) & obs["well_uid"].isin(set(wells.loc[model, "well_uid"]))]
coverage = (obs.groupby(["year", "campaign"])["valid"].sum() / model.sum()).sort_values().head(5)

edge = pd.DataFrame([
    ("wells more than 5 km from every district outline", int((wells["district_how"] == "none").sum()),
     "no district; kept, rain still read at the well"),
    ("wells whose CGWB state label disagrees with the coordinates", int((~wells["state_agrees"] & (wells["district_how"] != "none")).sum()),
     "kept at the coordinates: their water levels follow the neighbours there (README A5)"),
    ("land grid cells outside every district outline", int((land["n_districts"] == 0).sum()),
     "beyond the drawn border; rain kept at cell level"),
    ("districts with no land grid cell (islands)", 724 - dist.groupby(["district", "state"]).ngroups,
     "no rain; outside the IMD land mask"),
    ("wells with no Sy class from the map", int((~wells["sy_source"].isin(["published", "hydro_map", "hydro_map_nearest"])).sum()),
     "keep the ingest's state-median Sy, labelled in sy_source"),
], columns=["case", "count", "handling"])
print("thinnest survey rounds (valid readings per modelling well):")
print(coverage.round(3).to_string())
edge

thinnest survey rounds (valid readings per modelling well):
year  campaign
2020  May         0.010
2021  May         0.010
2012  Aug         0.028
2016  Jan         0.120
2020  Aug         0.409


,case,count,handling
0,wells more than 5 km from every district outline,7,"no district; kept, rain still read at the well"
1,wells whose CGWB state label disagrees with th...,489,kept at the coordinates: their water levels fo...
2,land grid cells outside every district outline,91,beyond the drawn border; rain kept at cell level
3,districts with no land grid cell (islands),4,no rain; outside the IMD land mask
4,wells with no Sy class from the map,0,"keep the ingest's state-median Sy, labelled in..."


## Checks

The notebook stops here if anything above went wrong.

In [10]:
w = pd.read_parquet(CORE_DIR / "well_master.parquet")
m = w[w["view_modelling"]]
ws = pd.read_parquet(CORE_DIR / "fact_well_season.parquet", columns=["well_uid", "sy_source", "storage_change_mm", "delta_h_m"])

assert len(w) == 32_299 and w["well_uid"].is_unique
assert int(w["view_strict"].sum()) == 2_759                       # the published set still reproduces
assert w["sy"].notna().all()
assert np.isclose(w.loc[w["sy_source"] == "published", "sy"], w.loc[w["sy_source"] == "published", "sy_ingest"]).all()
assert not m["aquifer"].isin(BLANK).any() and not m["well_type"].isin(BLANK).any()
assert (m.loc[m["aquifer_source"] == "inferred_dug_well", "well_type"].isin(DUG)).all()
assert set(ws["sy_source"].unique()) <= set(w["sy_source"].unique())   # season tables carry the new Sy
assert ws["storage_change_mm"].notna().sum() == ws["delta_h_m"].notna().sum()
assert len(full) == len(cell_season)

measured = m["sy_source"].isin(["published", "hydro_map", "hydro_map_nearest"]).mean()
print(f"all checks pass | modelling wells with a measured Sy class {measured:.1%} "
      f"| aquifer known or inferred {(m['aquifer'] != 'Unknown').mean():.1%}")

all checks pass | modelling wells with a measured Sy class 100.0% | aquifer known or inferred 94.6%


## Refresh the data dictionary

`core_dictionary.csv` is generated from the tables, and it fails if a column has no description.

In [11]:
sys.path.insert(0, str(REPO / "ml"))
from scripts import core_dictionary
assert core_dictionary.main() == 0

16 tables | 289 columns documented
wrote core_dictionary.csv and core_tables.csv to C:\Users\Public\BITS\BITS_grp18\data_cleaning\data
